# 04 — MLflow Judges & LLM Evaluation

**UI tab:** Evaluation · Judges

**LLM-as-a-Judge** uses a second LLM to score your model's outputs — useful when there is no single correct answer.

```
Evaluation pipeline
  Questions  →  gemini-2.5-flash  →  Answers
  Answers    →  gemini-2.5-pro    →  Scores (1–5)
  Scores     →  MLflow            →  Logged metrics
```

> Start the MLflow server first: `mlflow server --host 127.0.0.1 --port 5000`

In [ ]:
!pip install mlflow google-genai pandas --quiet

In [ ]:
import os, re, json
import pandas as pd
from google import genai
from google.genai import types
import mlflow
from mlflow.metrics import make_metric, MetricValue

os.environ["GOOGLE_API_KEY"] = "YOUR_GOOGLE_API_KEY_HERE"
client = genai.Client(api_key=os.environ["GOOGLE_API_KEY"])

mlflow.set_tracking_uri("http://127.0.0.1:5000")
mlflow.set_experiment("04-MLflow-Judges")

print("MLflow", mlflow.__version__, "ready")

## Step 1 — Eval dataset + generate predictions

In [ ]:
eval_data = pd.DataFrame({
    "question": [
        "What is MLflow?",
        "What is experiment tracking?",
        "What is a model registry?"
    ],
    "ground_truth": [
        "MLflow is an open-source platform for managing the end-to-end ML lifecycle.",
        "Experiment tracking records parameters, metrics and outputs for reproducibility.",
        "A model registry is a central store for versioning and managing ML models."
    ]
})

# Generate predictions with gemini-2.5-flash (the model under test)
eval_data["prediction"] = [
    (client.models.generate_content(
        model="gemini-2.5-flash",
        contents=[q],
        config=types.GenerateContentConfig(system_instruction="Answer in 1-2 sentences.")
    ).text or "").strip()
    for q in eval_data["question"]
]

print(eval_data[["question", "prediction"]].to_string(index=False))

## Step 2 — Gemini Pro as the judge

In [ ]:
JUDGE_PROMPT = """Rate this answer 1-5 for accuracy compared to the reference.
Question: {q}
Reference: {ref}
Answer: {ans}
Reply ONLY with JSON: {{"score": <int>, "reason": "<one sentence>"}}"""

def judge(question, ground_truth, prediction):
    prompt = JUDGE_PROMPT.format(q=question, ref=ground_truth, ans=prediction)
    response = client.models.generate_content(
        model="gemini-2.5-pro",
        contents=[prompt],
        config=types.GenerateContentConfig(temperature=0.0, max_output_tokens=200)
    )
    text = (response.text or "").strip()
    match = re.search(r'\{.*\}', text, re.DOTALL)
    if match:
        data = json.loads(match.group())
        return data.get("score", 3), data.get("reason", "")
    return 3, "parse error"

scores, reasons = [], []
for _, row in eval_data.iterrows():
    score, reason = judge(row["question"], row["ground_truth"], row["prediction"])
    scores.append(score)
    reasons.append(reason)
    print(f"Score: {score}/5 | {reason}")

eval_data["score"] = scores
eval_data["reason"] = reasons

## Step 3 — Log evaluation results to MLflow

In [ ]:
import tempfile

with mlflow.start_run(run_name="gemini-judge-eval"):
    mlflow.log_params({
        "model_under_test": "gemini-2.5-flash",
        "judge_model": "gemini-2.5-pro",
        "num_samples": len(eval_data)
    })
    mlflow.log_metrics({
        "avg_score": eval_data["score"].mean(),
        "min_score": eval_data["score"].min(),
        "max_score": eval_data["score"].max()
    })

    # Save detailed results as a CSV artifact
    with tempfile.NamedTemporaryFile(mode="w", suffix=".csv", delete=False) as f:
        eval_data.to_csv(f, index=False)
        tmp_path = f.name
    mlflow.log_artifact(tmp_path, artifact_path="evaluation")

    print(f"Average score: {eval_data['score'].mean():.2f} / 5")
    print("Results logged. Check Artifacts tab for the full CSV.")

## Step 4 — `make_genai_metric` (reusable custom judge metric)

In [ ]:
# Define a reusable conciseness metric using Gemini as judge
def conciseness_fn(predictions, inputs, targets=None, **kwargs):
    scores = []
    for q, pred in zip(inputs, predictions):
        prompt = (
            f"Rate how concise this answer is (1=very verbose, 5=perfectly concise).
"
            f"Q: {q}
A: {pred}
Reply with ONE digit 1-5."
        )
        response = client.models.generate_content(
            model="gemini-2.5-pro",
            contents=[prompt],
            config=types.GenerateContentConfig(temperature=0.0, max_output_tokens=20)
        )
        text = (response.text or "").strip()
        digit = re.search(r'[1-5]', text)
        scores.append(int(digit.group()) if digit else 3)
    return MetricValue(scores=scores, aggregate_results={"mean": sum(scores)/len(scores)})

conciseness_metric = make_metric(eval_fn=conciseness_fn, greater_is_better=True, name="conciseness")

# Test the metric
with mlflow.start_run(run_name="conciseness-metric"):
    mlflow.log_param("judge", "gemini-2.5-pro")

    result = conciseness_metric(
        predictions=eval_data["prediction"].tolist(),
        inputs=eval_data["question"].tolist()
    )
    mlflow.log_metric("avg_conciseness", result.aggregate_results["mean"])

    print(f"Conciseness scores: {result.scores}")
    print(f"Average: {result.aggregate_results['mean']:.2f} / 5")

## MLflow UI — What to explore
```
04-MLflow-Judges experiment
├── gemini-judge-eval    → Params, metrics, Artifacts → evaluation/
└── conciseness-metric   → avg_conciseness metric

Select both runs → Compare → see metric differences
```
**Next →** `05_mlflow_datasets.ipynb`